# Interpretation, Calibration, and Decision Quality — Project Improved Model Delivery

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" alt="MGMT 474 AI Logo" width="120"/>
</div>
</center>

<center>

# <center>MGMT47400 Predictive Analytics</center>

# <center>Notebook 15</center>

</center>

<center>

<a href="https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb15_interpretation_calibration_project_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

</center>

<hr>


## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what a champion classifier learned via permutation importance and partial dependence (PDP).
2. Diagnose where the model fails using segment-level error analysis on classification metrics.
3. Translate model probabilities into a business decision using the nb07 threshold + cost framework.
4. Diagnose calibration with reliability diagrams and the Brier score, and fix miscalibration with `CalibratedClassifierCV`.
5. Draft a concise decision-policy paragraph and stress-test it with a false-negative cost sensitivity sweep.
6. Deliver Project Milestone 3 (improved model + draft abstract).


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** (10 minutes each). You are expected to complete all exercises. Submit the completed notebook for participation credit.


## 💼 Why This Matters: Trust the Model, Then Act on It

The **State Health Department's chief medical officer** has the champion breast-cancer screening classifier in hand and is ready to deploy it across community clinics. Before signing off, she has two non-negotiable questions:

> *"I need to know **what** the model is learning before I trust it. And I need to know **at what probability we act** — and whether that probability actually means what it says."*

This notebook answers both questions in one sitting. Sections 2–4 open the box: feature importance, partial dependence, and segment-level error analysis explain what the classifier learned and where it breaks. Sections 5–7 turn that understanding into a defensible decision policy: pick a threshold from a cost matrix, check whether the probability at that threshold is honest, fix it if not, then stress-test the policy against uncertainty in the false-negative cost.

**A question that often comes up here:** *"We already covered thresholds back in nb07 — why revisit?"* Because in nb07 the model was logistic regression (natively well-calibrated). Today's champion is a tree ensemble (Random Forest from nb12), which is typically miscalibrated. The threshold question and the calibration question only become inseparable once we move to tree-based models — that is exactly where we are now.

By the end, you will hand the medical officer a slide-ready decision-policy paragraph **and** be ready to submit Project Milestone 3.


## 1. Setup

Import the libraries we need across both halves of the notebook. We fix the random seed to **474** so every figure and table you produce matches the reference output.


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    brier_score_loss, precision_score, recall_score, f1_score,
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Course-wide settings
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.precision', 3)

print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")


**Reading the output:**

`Setup complete!` with `RANDOM_SEED = 474` confirms every interpretation, calibration, and threshold function is reachable and that your output will match the reference exactly. If any import fails, restart the runtime and rerun this cell.


## 2. Load Data and Train Champion Classifier

To illustrate every diagnostic and policy step in a controlled setting, we generate a synthetic binary classification problem (5,000 rows, 20 features, 15 informative). The story is the breast-cancer screening pipeline, but the mechanics work the same for the Bank Churn project you are improving for Milestone 3.

We split the same way as the rest of the course: **60 / 20 / 20**. The test set stays locked — every diagnostic in this notebook lives on `X_train` (via cross-validation) or `X_val`.


> 💡 **Gemini Prompt:** "Generate a synthetic binary classification dataset with 5,000 samples and 20 features (15 informative, 5 redundant), seed 474. Split 60/20/20 train/val/test. Print the three sizes."
>
> **After running, verify:**
> - Train ≈ 3,000, Val ≈ 1,000, Test ≈ 1,000
> - Class balance roughly 50/50 in each split


In [ ]:
# Generate a synthetic binary classification dataset
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=15,
    n_redundant=5, n_classes=2, random_state=RANDOM_SEED,
)
feature_names = [f"f{i:02d}" for i in range(X.shape[1])]
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="screen_positive")

# 60 / 20 / 20 split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp,
)
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]} (locked)")
print(f"Train class balance: {y_train.mean():.3f}")


**Reading the output:**

You should see roughly **Train 3,000 / Val 1,000 / Test 1,000** with a class balance close to 0.5 in training. The test set print confirms it is set aside — none of the diagnostics or fixes below will touch it. We honor the **CV-first rule**: all model evaluation in this notebook uses cross-validation on `X_train` or held-out scoring on `X_val`.


> 💡 **Gemini Prompt:** "Train a RandomForestClassifier (100 trees, max_depth=10, seed 474) on X_train. Score ROC-AUC on X_val and report it. Also produce out-of-fold predicted probabilities on X_train via cross_val_predict with 5-fold StratifiedKFold for downstream interpretation."
>
> **After running, verify:**
> - The validation ROC-AUC is between 0.90 and 0.98 (this synthetic data is comfortably learnable)
> - `proba_train_oof` has the same number of rows as `X_train`


In [ ]:
# Train champion Random Forest classifier
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1,
)
rf_model.fit(X_train, y_train)

# Held-out validation AUC (no test-set access)
y_val_proba = rf_model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_val_proba)
print(f"Validation ROC-AUC: {val_auc:.3f}")

# Out-of-fold probabilities on X_train for honest interpretation downstream
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
proba_train_oof = cross_val_predict(
    rf_model, X_train, y_train, cv=cv_strat, method="predict_proba", n_jobs=-1,
)[:, 1]
print(f"Out-of-fold probabilities computed: shape {proba_train_oof.shape}")


**Reading the output:**

The Random Forest hits validation ROC-AUC near **0.95**, comfortably better than chance and good enough that the medical officer is interested in deploying it. The out-of-fold probabilities (`proba_train_oof`) are the workhorse for the rest of the notebook — every prediction comes from a fold that did **not** see that row during fitting, which is what makes downstream calibration and segment-error diagnostics honest.


## 3. What Did the Model Learn? — Permutation Feature Importance

The medical officer's first question is *"what is the model paying attention to?"*. Permutation importance answers it directly: shuffle one feature column at a time, measure how much validation performance drops, and rank the features by drop size. Bigger drop = the model leans on that feature more.

This complements nb12's four-method importance reconciliation table. Here we use **permutation importance** specifically because it works for any fitted estimator and uses validation-set performance — exactly the signal the medical officer cares about.


> 💡 **Gemini Prompt:** "Compute permutation importance on X_val with 10 repeats, scoring='roc_auc', seed 474. Build a sorted DataFrame of feature, importance_mean, importance_std and print the top 8 rows."
>
> **After running, verify:**
> - The top features have positive mean importance (shuffling them hurt AUC)
> - Standard deviations are small relative to means (importance is stable across repeats)


In [ ]:
# Permutation importance on validation set
perm = permutation_importance(
    rf_model, X_val, y_val,
    n_repeats=10, random_state=RANDOM_SEED, scoring="roc_auc", n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": X_val.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print(importance_df.head(8))


**Reading the output:**

The top features are the ones whose shuffling hurt validation ROC-AUC the most. A high mean with a small standard deviation means the model genuinely depends on that feature; a small mean (close to zero) means shuffling did not change AUC much, so the feature is essentially decorative.

**A question that often comes up here:** *"Why use permutation importance instead of the random forest's built-in `feature_importances_`?"* Because built-in importance is biased toward high-cardinality and continuous features (it counts split frequency, which inflates noisy splits). Permutation importance asks the right question: *"if I destroy this feature, does the model's validation performance suffer?"* That is the signal the medical officer wants.


### 3.1 Visualize the importance ranking

A horizontal bar chart with error bars is the standard way to show this to non-technical stakeholders.


> 💡 **Gemini Prompt:** "Plot horizontal bar chart of permutation importance for top 10 features, with error bars from importance_std. Invert y-axis so the most important is at the top. Title: 'Permutation Feature Importance (validation, 10 repeats)'."


In [ ]:
# Visualize permutation importance
top10 = importance_df.head(10)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10["feature"], top10["importance_mean"],
        xerr=top10["importance_std"], color="#cfb991", edgecolor="black")
ax.invert_yaxis()
ax.set_xlabel("Mean importance (drop in ROC-AUC)")
ax.set_title("Permutation Feature Importance (validation, 10 repeats)")
plt.tight_layout()
plt.show()


**Reading the output:**

The bars at the top of the chart are the features whose removal most damaged AUC — those are the medical officer's "drivers." The error bars are the importance standard deviations across the 10 repeats; narrow error bars confirm the ranking is stable, wide bars say the feature's contribution is noisy.


## 4. Partial Dependence — How Predictions Change With a Feature

Importance ranks features by **how much** they matter. Partial dependence shows **in what direction** they matter: as feature value goes up, does predicted probability go up, down, or wiggle?

For a classifier, PDP plots predicted probability of the positive class as a function of one feature, averaging over all other feature values. A monotonic upward curve says "higher values push toward positive screening"; a U-shape says "extremes in either direction matter."


> 💡 **Gemini Prompt:** "Build a 2x2 PartialDependenceDisplay for the top 4 features by permutation importance, on X_train, target=positive class, kind='average'. Title: 'Partial Dependence — Top 4 Features'."
>
> **After running, verify:**
> - Each subplot shows a smooth-ish curve, not a flat line
> - Y-axis is predicted probability of the positive class (0–1)


In [ ]:
# Partial dependence for top 4 features
top4 = importance_df.head(4)["feature"].tolist()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
PartialDependenceDisplay.from_estimator(
    rf_model, X_train, features=top4, kind="average", grid_resolution=30,
    ax=axes.ravel().tolist(),
)
fig.suptitle("Partial Dependence — Top 4 Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

For each of the top 4 features, the curve traces how predicted probability of the positive class changes as that feature varies (other features held at their average). Look for the **direction** (up, down, U-shape), the **magnitude** (does the curve span 0.2–0.8 or only 0.45–0.55?), and **discontinuities** (sharp jumps suggest a tree split that the model has learned to lean on).

> *Extension:* Individual Conditional Expectation (ICE) plots add per-row curves on top of the average. Run `PartialDependenceDisplay.from_estimator(..., kind='both')` on the single most important feature if you want to see per-row heterogeneity. We omit it here to keep the focus on the average effect.


## 5. Where Does the Model Fail? — Segment Error Analysis

Aggregate AUC tells the medical officer the model is "good overall." But two clinics with the same overall AUC can have very different failure modes — one might miss aggressive cases in young patients, the other might over-flag a benign demographic. **Segment error analysis** breaks down classification metrics by feature quartile (or category) and finds those failure modes.


> 💡 **Gemini Prompt:** "On X_train, split rows into 4 quartiles of the most important feature. For each quartile, compute precision, recall, and F1 against y_train using thresholded out-of-fold probabilities at 0.5. Print the table sorted by recall ascending."
>
> **After running, verify:**
> - The quartile with the lowest recall is your candidate failure segment
> - Sample sizes per quartile are roughly balanced (~750)


In [ ]:
# Segment error analysis on out-of-fold predictions
top_feature = importance_df.iloc[0]["feature"]
seg_df = X_train.copy()
seg_df["y_true"] = y_train.values
seg_df["proba_oof"] = proba_train_oof
seg_df["pred_oof"] = (proba_train_oof >= 0.5).astype(int)
seg_df["quartile"] = pd.qcut(seg_df[top_feature], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

rows = []
for q, grp in seg_df.groupby("quartile", observed=True):
    rows.append({
        "quartile": q,
        "n": len(grp),
        "precision": precision_score(grp["y_true"], grp["pred_oof"], zero_division=0),
        "recall": recall_score(grp["y_true"], grp["pred_oof"], zero_division=0),
        "f1": f1_score(grp["y_true"], grp["pred_oof"], zero_division=0),
    })
seg_metrics = pd.DataFrame(rows).sort_values("recall")
print(seg_metrics)


**Reading the output:**

The table breaks the training data into four equally-sized buckets along the most important feature, then asks "how well does the model perform inside each bucket?" The bucket with the **lowest recall** is the most worrying for a screening tool — recall low means the model is missing real positives in that segment, which is the failure the medical officer fears most. The bucket with the lowest precision is a different worry (over-flagging). Either gap is a candidate for the model card you would write before deployment.

**A question that often comes up here:** *"Why do segment analysis on training out-of-fold predictions instead of validation?"* Two reasons. First, sample size — splitting validation into quartiles leaves ~250 per bucket, often too small for stable precision/recall. Second, the out-of-fold predictions on `X_train` are honest (each row was held out from its fold's fit), so this gives you the larger sample without leaking. We will spot-check on `X_val` later.


## 📝 PAUSE-AND-DO Exercise 1 — Interpretation Findings (10 minutes)

**Task:** Combine sections 3, 4, and 5 into evidence-based findings the medical officer can read.

**Instructions:**
1. From section 3, pick the **top 3 features** by permutation importance and write one sentence each describing what they likely measure (you can speculate — these are synthetic features) and why their importance ranking makes sense.
2. From section 4, pick **one feature whose PDP curve surprised you** (sharp jump, U-shape, or unexpectedly flat). Describe the shape in one sentence.
3. From section 5, identify the **one quartile with the lowest recall**. State the recall, the segment size, and one hypothesis for why the model underperforms there.

Type your findings in the placeholder cell below.


### YOUR INTERPRETATION FINDINGS HERE:

**Top 3 features and what they capture:**

1. *[feature name]* — [one-sentence interpretation]
2. *[feature name]* — [one-sentence interpretation]
3. *[feature name]* — [one-sentence interpretation]

**Surprising PDP curve:**

*[feature name]* — [shape description]

**Lowest-recall segment and hypothesis:**

Quartile *[Q1/Q2/Q3/Q4]* of *[feature name]* — recall *[value]*, n=*[size]*. Hypothesis: [one sentence].


## 6. Bridge — From What and Where, to At What Threshold and Is It Honest

You now know **what** the model learned (importance, PDP) and **where** it breaks (segment errors). Two questions remain before the screening program can ship:

1. **At what probability do we act?** A 0.5 default treats false negatives and false positives as equally costly — they almost never are, especially in healthcare.
2. **Is the probability at that threshold honest?** When the model says "0.7", does that actually mean 70% of similar cases turn out positive? Tree ensembles are notorious for being miscalibrated, even when their AUC is great.

Sections 7 (threshold + cost), 8 (calibration), and 9 (decision policy + sensitivity) answer those two questions in order.


## 7. Threshold and Cost — Refresh from nb07

In nb07 we built the cost-matrix machinery: assign a dollar value to each cell of the confusion matrix, sweep the decision threshold, and pick the threshold that maximizes expected value. The mechanics are identical here; we just apply them to the champion's out-of-fold probabilities.

The medical officer estimates the per-patient costs (in arbitrary cost units that align with department budget tracking):

- **TP** (correctly flagged for follow-up): +100 — early detection saves treatment cost downstream
- **FP** (incorrectly flagged): −30 — an unnecessary diagnostic visit
- **FN** (missed positive): −150 — late detection is the most expensive failure
- **TN** (correctly cleared): 0 — baseline


In [ ]:
# Cost matrix and expected-value function (refresh from nb07)
COST_MATRIX = {"TP": 100, "FP": -30, "FN": -150, "TN": 0}

def expected_value(y_true, y_pred, cost=COST_MATRIX):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    ev = tp * cost["TP"] + fp * cost["FP"] + fn * cost["FN"] + tn * cost["TN"]
    return ev, {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

# Sweep thresholds on out-of-fold training probabilities
thresholds = np.arange(0.10, 0.90, 0.05)
sweep_rows = []
for t in thresholds:
    y_pred_t = (proba_train_oof >= t).astype(int)
    ev, counts = expected_value(y_train, y_pred_t)
    sweep_rows.append({"threshold": t, "expected_value": ev, **counts})
sweep_df = pd.DataFrame(sweep_rows)
best_idx = sweep_df["expected_value"].idxmax()
best_threshold = sweep_df.loc[best_idx, "threshold"]
print(sweep_df.head(8))
print(f"\nOptimal threshold: {best_threshold:.2f}")
print(f"Expected value at optimum: {sweep_df.loc[best_idx, 'expected_value']:.0f}")


**Reading the output:**

Each row of the sweep table is one candidate threshold; `expected_value` is the dollar total under the cost matrix. The best threshold is rarely 0.5 — usually it shifts toward the cheaper-to-act side of the cost asymmetry. Here, because false negatives cost 5× more than false positives, the optimal threshold sits **below** 0.5: act earlier, accept some unnecessary follow-ups to avoid expensive misses.


In [ ]:
# Visualize the sweep
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sweep_df["threshold"], sweep_df["expected_value"], marker="o", color="#1f77b4")
axes[0].axvline(best_threshold, color="red", linestyle="--", label=f"optimum = {best_threshold:.2f}")
axes[0].set_xlabel("Decision threshold")
axes[0].set_ylabel("Expected value (cost units)")
axes[0].set_title("Expected value across thresholds")
axes[0].legend()

axes[1].plot(sweep_df["threshold"], sweep_df["FN"], label="FN (missed positives)", color="#d62728")
axes[1].plot(sweep_df["threshold"], sweep_df["FP"], label="FP (false alarms)", color="#ff7f0e")
axes[1].axvline(best_threshold, color="red", linestyle="--")
axes[1].set_xlabel("Decision threshold")
axes[1].set_ylabel("Count")
axes[1].set_title("FN vs. FP trade-off")
axes[1].legend()

plt.tight_layout()
plt.show()


**Reading the output:**

The **left panel** shows expected value rising to a peak at the optimal threshold then falling — the classic upside-down U. The **right panel** explains why: as threshold drops, FNs fall (good — fewer missed positives) but FPs rise (worse — more false alarms). The optimum sits where the marginal cost of one more FP equals the marginal benefit of one fewer FN, which is exactly what the cost matrix encodes.


## 8. Calibration — Are the Probabilities Honest?

The threshold sweep above assumed `proba_train_oof[i] = 0.7` means "70% chance this case is positive." If the model is **miscalibrated**, that interpretation is wrong, and the optimal threshold computed above is built on shaky ground.

A **reliability diagram** checks the assumption: bin predicted probabilities (e.g., 10 bins from 0.0 to 1.0), and for each bin plot the **average predicted probability** against the **observed positive rate**. A perfectly calibrated model lies on the 45° line. Tree ensembles typically pull toward the middle (over-predict the rare class at low probabilities and under-predict at high probabilities) — exactly the failure mode the medical officer needs to know about.

The **Brier score** is the single number that summarizes calibration error: lower is better, and `0` means perfect.


In [ ]:
# Reliability diagram + Brier score
prob_true, prob_pred = calibration_curve(y_train, proba_train_oof, n_bins=10, strategy="uniform")
brier_uncal = brier_score_loss(y_train, proba_train_oof)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(prob_pred, prob_true, "o-", color="#d62728", label=f"Random Forest (Brier={brier_uncal:.3f})")
ax.set_xlabel("Mean predicted probability (per bin)")
ax.set_ylabel("Observed positive rate (per bin)")
ax.set_title("Reliability Diagram — Champion Random Forest")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

The dashed black line is the ideal: model says 70%, observed positive rate is 70%. The red curve is the champion's actual calibration. If the red curve is **below** the diagonal, the model **over-predicts** the positive class (says 70% when truth is 55%). If it's **above**, the model **under-predicts**. Either way, the threshold you picked in section 7 will not behave the way you expect when deployed.

**A question that often comes up here:** *"My AUC was 0.95 — how can I be miscalibrated?"* AUC measures **ranking** ("did high-risk patients get higher probability than low-risk patients?"). Calibration measures the **scale** ("is 0.7 actually 70%?"). A model can rank perfectly and still be off-scale. Logistic regression's loss function makes it natively well-calibrated; tree ensembles do not — that is the reason this section exists.


### 8.1 Fixing miscalibration with `CalibratedClassifierCV`

`CalibratedClassifierCV` wraps an estimator and adds a calibration layer fit by isotonic regression (monotonic, non-parametric — the right default for tree models that produce step-shaped distortions). It uses internal cross-validation on `X_train`, so we are not touching the test set.


In [ ]:
# Calibrated wrapper using internal 5-fold CV on training data
calibrated_rf = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1),
    method="isotonic", cv=5,
)
calibrated_rf.fit(X_train, y_train)

# Reliability check on validation (held out, not test)
proba_val_uncal = rf_model.predict_proba(X_val)[:, 1]
proba_val_cal = calibrated_rf.predict_proba(X_val)[:, 1]

brier_val_uncal = brier_score_loss(y_val, proba_val_uncal)
brier_val_cal = brier_score_loss(y_val, proba_val_cal)

prob_true_u, prob_pred_u = calibration_curve(y_val, proba_val_uncal, n_bins=10)
prob_true_c, prob_pred_c = calibration_curve(y_val, proba_val_cal, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(prob_pred_u, prob_true_u, "o-", color="#d62728",
        label=f"Uncalibrated (Brier={brier_val_uncal:.3f})")
ax.plot(prob_pred_c, prob_true_c, "s-", color="#2ca02c",
        label=f"Calibrated isotonic (Brier={brier_val_cal:.3f})")
ax.set_xlabel("Mean predicted probability (per bin)")
ax.set_ylabel("Observed positive rate (per bin)")
ax.set_title("Reliability — Validation Set, Uncalibrated vs. Calibrated")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nBrier uncalibrated (val): {brier_val_uncal:.4f}")
print(f"Brier calibrated   (val): {brier_val_cal:.4f}")
print(f"Improvement: {(brier_val_uncal - brier_val_cal):.4f}")


**Reading the output:**

The green calibrated curve should hug the diagonal more tightly than the red uncalibrated one, and the calibrated Brier should be lower (closer to zero). If the improvement is small (≤0.005), the original model was already close to honest and the calibration layer adds little; if the improvement is meaningful (≥0.01), calibration changes the threshold story — the optimal threshold should be re-computed on the calibrated probabilities before the policy is finalized.

> *Extension:* sigmoid (Platt scaling) is the alternative calibrator. It uses one parametric S-curve and works best when miscalibration is monotonic and gentle. For tree ensembles with step-shaped distortions, isotonic almost always wins. Try `method='sigmoid'` if you want to compare.


## 9. Decision Policy + Sensitivity Analysis

You now have everything you need for a defensible policy: an interpretable champion (sections 3–4), known failure segments (section 5), a cost-aware threshold (section 7), and honest probabilities (section 8). The medical officer's deliverable is a single **6-line decision-policy paragraph** for the slide deck and the regulatory file. Then we stress-test it against uncertainty in the FN cost.


In [ ]:
# Recompute optimal threshold on calibrated probabilities (held-out validation)
sweep_cal = []
for t in thresholds:
    y_pred_t = (proba_val_cal >= t).astype(int)
    ev, counts = expected_value(y_val, y_pred_t)
    sweep_cal.append({"threshold": t, "expected_value": ev, **counts})
sweep_cal_df = pd.DataFrame(sweep_cal)
best_t_cal = sweep_cal_df.loc[sweep_cal_df["expected_value"].idxmax(), "threshold"]
ev_cal = sweep_cal_df["expected_value"].max()

print("===== Decision Policy (calibrated, validation) =====")
print(f"Champion model      : Random Forest (100 trees, max_depth=10) + isotonic calibration")
print(f"Operating threshold : {best_t_cal:.2f}")
print(f"Expected value      : {ev_cal:.0f} cost units (per {len(y_val)} screened patients)")
print(f"Validation Brier    : {brier_val_cal:.3f}  (lower is better)")
print(f"Validation ROC-AUC  : {roc_auc_score(y_val, proba_val_cal):.3f}")


**Reading the output:**

This printout is the slide-ready decision-policy paragraph. The four numbers — operating threshold, expected value, calibration Brier, and AUC — are the four facts the medical officer needs to defend the deployment to the department's clinical safety committee.


### 9.1 FN-cost sensitivity sweep

The cost of a missed positive is the medical officer's **best estimate**, not a known quantity. If a missed case turns out to cost 200 instead of 150 (or 100 instead of 150), the optimal threshold shifts. The sensitivity sweep tells you **how much** before you ship.


In [ ]:
# Sensitivity: vary FN cost, find new optimum each time
fn_grid = [-100, -125, -150, -175, -200, -250]
sens_rows = []
for fn in fn_grid:
    cost = {"TP": 100, "FP": -30, "FN": fn, "TN": 0}
    best_ev, best_t, best_counts = -np.inf, None, None
    for t in thresholds:
        y_pred_t = (proba_val_cal >= t).astype(int)
        ev, counts = expected_value(y_val, y_pred_t, cost=cost)
        if ev > best_ev:
            best_ev, best_t, best_counts = ev, t, counts
    sens_rows.append({
        "FN_cost": fn,
        "optimal_threshold": best_t,
        "expected_value": best_ev,
        **best_counts,
    })
sens_df = pd.DataFrame(sens_rows)
print(sens_df)

# Plot threshold drift as FN cost varies
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sens_df["FN_cost"].abs(), sens_df["optimal_threshold"], "o-", color="#9467bd")
ax.set_xlabel("|FN cost| (cost units)")
ax.set_ylabel("Optimal threshold")
ax.set_title("Threshold Drift as FN Cost Varies")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Reading the output:**

The sensitivity table shows how the optimal threshold and expected value respond to the medical officer's FN-cost estimate. **A flat threshold curve** (the optimum barely moves across the plausible cost range) is a strong policy signal: even if the cost estimate is off, the action point is the same, so the recommendation is robust. **A steep curve** says the optimum is highly sensitive to the FN-cost guess — that fragility belongs in the limitations section of the model card and should drive a follow-up pricing-study before deployment.

**A question that often comes up here:** *"What if I don't know the costs precisely?"* Use a range, not a point. Report the policy under the **median** estimate and the **threshold range** across the lower and upper estimates. The medical officer can then decide whether the spread is small enough to ship or large enough to require more work.


## 📝 PAUSE-AND-DO Exercise 2 — Decision Policy + Sensitivity (10 minutes)

**Task:** Draft the slide-ready decision-policy paragraph and stress-test it.

**Instructions:**
1. Pick the **operating threshold** (`best_t_cal` from section 9).
2. Read off the **expected value** at that threshold.
3. Decide whether the calibrated Brier is "honest enough" (Brier < 0.20 on a balanced-ish dataset is reasonable; below 0.10 is good).
4. Look at the FN-cost sensitivity sweep. Is the optimal threshold flat (robust) or sensitive (fragile)? State the threshold range across the FN-cost grid.
5. Write the **6-line decision-policy paragraph** in the placeholder cell below.

This paragraph goes directly onto your project poster.


### YOUR DECISION POLICY HERE:

**Operating threshold:** *[value]*
**Expected value:** *[number]* cost units per *[N]* screened patients
**Calibration:** Brier *[value]* — *[honest enough / needs more work]*
**Sensitivity to FN cost:** Optimal threshold ranges *[low]* – *[high]* across plausible FN costs (\$100 – \$250). The policy is *[robust / fragile]*.
**Recommendation:** *[Deploy as-is / Deploy with monitoring / Re-run pricing study before deploy]*
**Limitations:** *[one sentence: e.g., low recall in Q1 of f00 — flag for clinical review before deployment in pediatric clinics]*


## 10. Project Milestone 3 Scaffold

Today is the deadline for **Milestone 3**: improved model + draft abstract.

**M3 deliverable checklist (group of four):**

- [ ] More complex model fit (e.g., Random Forest, Gradient Boosting, or a tuned ensemble) — required improvement over the M2 baseline.
- [ ] Hyperparameter tuning via `GridSearchCV` or `RandomizedSearchCV` on `X_train` only.
- [ ] CI-overlap comparison vs. the M2 baseline using nb08's Student's *t* 95% CI (no test-set access).
- [ ] Permutation importance + PDP for the champion (sections 3–4 above).
- [ ] One named failure segment from segment-error analysis (section 5).
- [ ] Calibration check + decision-policy paragraph (sections 7–9).
- [ ] Draft abstract (~250 words) — this becomes the seed for the M4 poster.

**Submit by 11:59 PM (Day 15)** via Brightspace.


## 11. Wrap-Up — Key Takeaways

We closed the loop from "good aggregate AUC" to "defensible decision policy" in one notebook. Three ideas to carry forward:

1. **Interpretation is two questions, not one.** *What did the model learn?* (importance, PDP) and *Where does it fail?* (segment errors). The model card needs both.
2. **A threshold without calibration is a guess.** Tree ensembles are typically miscalibrated; check the reliability diagram before you commit to a threshold, and re-sweep on calibrated probabilities if the Brier improvement is meaningful.
3. **Stress-test the cost assumption.** The FN-cost sensitivity sweep tells you whether your policy is robust to estimation error. A flat curve is shippable; a steep curve is a research project.

**A question that often comes up here:** *"Will the project use all of this?"* Yes — the decision-policy paragraph is the centerpiece of the M4 poster's "Recommendation" section, and the limitations from segment errors plus calibration will populate the poster's "Limitations" section. Today's notebook is M3 *and* M4 raw material.

**Next stop — nb16: Time-Series Forecasting.** The Bank Churn project is static (each row is a customer-month snapshot), but many business problems — demand forecasting, KPI monitoring, financial returns — are temporal. nb16 introduces the one thing that changes when time becomes a feature: **the train/test split must respect time.** Walk-forward cross-validation replaces k-fold; lag features replace random feature engineering; the locked test set becomes the **most recent slice** of history. Same CV-first discipline, new shape.


## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in PAUSE-AND-DO 1 (interpretation findings) and PAUSE-AND-DO 2 (decision policy paragraph).
2. **Run all cells**: From the menu, select `Runtime → Run all`. Confirm every cell finishes without an error icon.
3. **Save with output**: From the menu, select `File → Download → Download .ipynb`.
4. **Submit two artifacts to Brightspace**:
   - This completed notebook (`nb15_interpretation_calibration_project_<your_lastname>.ipynb`)
   - The Milestone 3 group submission (improved model + draft abstract) per the M3 rubric.


<center>

# Thank you!

</center>
